## Example 1: tblib functionality test 1

In [ ]:
import tblib
import sys
import pickle

# Example 1: Basic traceback capture and serialization
def nested_function():
    """Function that raises an exception for testing"""
    raise ValueError("This is a test error from nested function")

def outer_function():
    """Outer function that calls nested function"""
    nested_function()

print("Basic tblib traceback capture:")
try:
    outer_function()
except Exception:
    # Capture the current exception traceback
    exc_type, exc_value, exc_traceback = sys.exc_info()
    tb = tblib.Traceback(exc_traceback)
    
    print(f"Traceback captured: {type(tb)}")
    print(f"Has as_traceback method: {hasattr(tb, 'as_traceback')}")
    print(f"Can be pickled: {tb is not None}")
    
    # Show that it can be serialized
    serialized = pickle.dumps(tb)
    print(f"Serialized traceback size: {len(serialized)} bytes")
    
    # The handler IS the example (the try provokes the error on purpose), so it asserts what tblib did:
    # the pickled traceback must come back as a real traceback with the same frames.
    assert isinstance(exc_value, ValueError), repr(exc_value)
    frame_names = []
    restored = pickle.loads(serialized).as_traceback()
    while restored is not None:
        frame_names.append(restored.tb_frame.f_code.co_name)
        restored = restored.tb_next
    print(f"Round-tripped frames: {frame_names}")
    assert frame_names[-2:] == ["outer_function", "nested_function"], frame_names
else:
    raise AssertionError("outer_function() did not raise, so tblib captured nothing")

## Example 2: tblib functionality test 2

In [ ]:
import tblib
import pickle
import sys

# Example 2: Traceback transmission across processes (simulation)
def simulate_remote_execution():
    """Simulate function execution that might happen in another process"""
    def problematic_function(data):
        if data < 0:
            raise RuntimeError(f"Negative value not allowed: {data}")
        return data * 2
    
    try:
        return problematic_function(-5)
    except Exception:
        # Capture and return serialized traceback
        exc_type, exc_value, exc_traceback = sys.exc_info()
        tb = tblib.Traceback(exc_traceback)
        
        return {
            'success': False,
            'error_type': exc_type.__name__,
            'error_message': str(exc_value),
            'traceback': pickle.dumps(tb)
        }

print("Cross-process traceback simulation:")
result = simulate_remote_execution()

if not result['success']:
    print(f"Error type: {result['error_type']}")
    print(f"Error message: {result['error_message']}")
    
    # Deserialize and examine the traceback
    tb = pickle.loads(result['traceback'])
    print(f"Deserialized traceback type: {type(tb)}")
    print(f"Traceback can be converted back: {hasattr(tb, 'as_traceback')}")

## Example 3: tblib functionality test 3

In [ ]:
import tblib
import sys
import traceback

# Example 3: Re-raising preserved tracebacks
def store_and_reraise_traceback():
    """Demonstrate storing traceback and re-raising later"""
    stored_tb = None
    stored_exc = None
    
    # First, capture an exception
    try:
        def deep_call_stack():
            def level_3():
                raise KeyError("Key not found in deep function")
            def level_2():
                level_3()
            def level_1():
                level_2()
            level_1()
        
        deep_call_stack()
    except KeyError as e:
        # Store both exception and traceback
        exc_type, exc_value, exc_traceback = sys.exc_info()
        stored_tb = tblib.Traceback(exc_traceback)
        stored_exc = e
        print("Exception and traceback stored")
    else:
        raise AssertionError("expected KeyError from deep_call_stack()")
    
    # Later, re-raise with original traceback
    if stored_tb:
        try:
            # Re-raise with preserved stack trace
            raise stored_exc.with_traceback(stored_tb.as_traceback())
        except KeyError as e:
            # The rebuilt traceback still reaches the frame that originally raised.
            frames = [frame.name for frame in traceback.extract_tb(e.__traceback__)]
            assert frames[-3:] == ["level_1", "level_2", "level_3"], frames
            assert e.args == ("Key not found in deep function",), e.args
            print("Re-raised exception with preserved traceback")
            print("Current exception info available")
        else:
            raise AssertionError("expected the stored KeyError to be re-raised")

print("Traceback preservation and re-raising:")
store_and_reraise_traceback()

## Example 4: tblib functionality test 4

In [ ]:
import tblib
import pickle
import concurrent.futures
import sys

# Example 4: Distributed computing traceback handling
class DistributedTaskRunner:
    """Simulate distributed task execution with traceback preservation"""
    
    @staticmethod
    def execute_task(task_data):
        """Execute a task that might fail"""
        task_id, operation, value = task_data
        
        try:
            if operation == "divide":
                if value == 0:
                    raise ZeroDivisionError("Cannot divide by zero")
                return {"task_id": task_id, "result": 100 / value, "success": True}
            elif operation == "sqrt":
                if value < 0:
                    raise ValueError("Cannot calculate sqrt of negative number")
                return {"task_id": task_id, "result": value ** 0.5, "success": True}
        except Exception as e:
            # Capture and serialize the traceback for remote debugging
            exc_type, exc_value, exc_traceback = sys.exc_info()
            tb = tblib.Traceback(exc_traceback)
            
            return {
                "task_id": task_id,
                "success": False,
                "error": str(e),
                "error_type": exc_type.__name__,
                "traceback": pickle.dumps(tb)
            }

print("Distributed computing with traceback preservation:")

# Simulate multiple tasks
tasks = [
    (1, "divide", 10),
    (2, "divide", 0),    # Will fail
    (3, "sqrt", 16),
    (4, "sqrt", -4),     # Will fail
]

runner = DistributedTaskRunner()
results = []

for task in tasks:
    result = runner.execute_task(task)
    results.append(result)
    
    if not result['success']:
        print(f"Task {result['task_id']} failed: {result['error']}")
        # Could deserialize traceback for debugging
        tb = pickle.loads(result['traceback'])
        print(f"  Traceback preserved: {type(tb)}")

print(f"Processed {len(results)} tasks with full error tracking")

## Example 5: tblib functionality test 5

In [ ]:
import tblib
import pickle
import sys

# Example 5: Production patterns and advanced tblib usage
class ErrorReportingSystem:
    """Production-ready error reporting with tblib"""
    
    def __init__(self):
        self.error_reports = []
    
    def capture_error(self, context, func, *args, **kwargs):
        """Execute function with comprehensive error capture"""
        try:
            return {"success": True, "result": func(*args, **kwargs)}
        except Exception as e:
            exc_type, exc_value, exc_traceback = sys.exc_info()
            tb = tblib.Traceback(exc_traceback)
            
            error_report = {
                "context": context,
                "error_type": exc_type.__name__,
                "error_message": str(exc_value),
                "traceback_data": pickle.dumps(tb),
                "args": args,
                "kwargs": kwargs
            }
            
            self.error_reports.append(error_report)
            return {"success": False, "error_id": len(self.error_reports) - 1}
    
    def debug_error(self, error_id):
        """Recreate error context for debugging"""
        if error_id < len(self.error_reports):
            report = self.error_reports[error_id]
            tb = pickle.loads(report["traceback_data"])
            
            print(f"Error in context: {report['context']}")
            print(f"Error type: {report['error_type']}")
            print(f"Error message: {report['error_message']}")
            print(f"Traceback available: {type(tb)}")
            return tb
        return None

# Test production error handling
print("Production error handling system:")
error_system = ErrorReportingSystem()

# Simulate various operations
def risky_calculation(x, y):
    if y == 0:
        raise ZeroDivisionError("Division by zero in calculation")
    return x / y

def risky_list_access(data, index):
    return data[index]

# Execute operations with error capture
results = [
    error_system.capture_error("calc_1", risky_calculation, 10, 2),
    error_system.capture_error("calc_2", risky_calculation, 10, 0),  # Error
    error_system.capture_error("list_1", risky_list_access, [1,2,3], 1),
    error_system.capture_error("list_2", risky_list_access, [1,2,3], 5),  # Error
]

print(f"Executed {len(results)} operations")
print(f"Captured {len(error_system.error_reports)} errors")

# Debug captured errors
for i, report in enumerate(error_system.error_reports):
    print(f"\nDebugging error {i}:")
    error_system.debug_error(i)